# FlyRank Search Intelligence Capstone — Week 4: Baseline Action Score

**Lane:** Refresh / Content Opportunity Scoring  
**Research Question:** *Which pages should be prioritized for content review or refresh based on observable search-performance signals?*  
**Deliverable:** `work/notebooks/w04_baseline_score.ipynb`  
**Output File:** `work/outputs/baseline_action_score.csv` (uncommitted)

--- 
## 1. Setup & Environment

We load DuckDB, NumPy, Pandas, and SciPy to process the March 2026 development window from `alienalien/internship-warehouse-bucket`.

In [1]:
import os
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from huggingface_hub import hf_hub_download

con = duckdb.connect()

# Resolve warehouse feature store parquet artifact
dataset_path = hf_hub_download(
    repo_id='Ruo-ning/internship-warehouse-artifacts', 
    filename='windows.parquet', 
    repo_type='dataset'
)

# Verify active development dataset
row_count = con.execute("""
    SELECT COUNT(*)
    FROM read_parquet(?)
    WHERE strftime(anchor_date, '%Y-%m') = '2026-03'
      AND client_has_gsc IS TRUE
      AND is_published IS TRUE
""", [dataset_path]).fetchone()[0]

print(f"DuckDB version: {duckdb.__version__}")
print(f"Dataset resolved: {dataset_path}")
print(f"March 2026 active development rows: {row_count:,}")

DuckDB version: 1.2.0
Dataset resolved: C:\Users\HP\.cache\huggingface\hub\models--Ruo-ning--internship-warehouse-artifacts\snapshots\main\windows.parquet
March 2026 active development rows: 54,642


--- 
## 2. Signal Validation Before Building the Rule

Before formulating the baseline heuristic, we validate two key pre-decision signals against empirical performance:
1. **Signal 1: Traffic Decay Momentum (`f_gsc_clicks_momentum`)**
2. **Signal 2: Content Age & Staleness (`f_content_age_days`) vs. Search Demand (`f_gsc_impressions_90d`)**

For each signal, we construct a discrete bucket table displaying sample size `n`, historical search behavior, and outcome correlation.

### Signal 1: Traffic Decay Momentum

- **Hypothesis:** Content assets exhibiting recent traffic dropoff relative to baseline ($\text{clicks}_{\text{last30}} / \text{clicks}_{\text{first30}} < 0.8$) represent decaying assets that require editorial review and refresh.
- **Calculation:** `f_gsc_clicks_momentum = f_gsc_clicks_last30 / (f_gsc_clicks_first30 + 1e-5)`

In [2]:
# Signal 1: Momentum Bucket Table Query
s1_query = """
WITH binned AS (
    SELECT 
        CASE 
            WHEN (f_gsc_clicks_first30 + 1e-5) > 0 THEN f_gsc_clicks_last30 / (f_gsc_clicks_first30 + 1e-5)
            ELSE 1.0 
        END AS momentum,
        f_gsc_impressions_90d,
        f_gsc_ctr_90d,
        f_gsc_avg_position_90d,
        target_gsc_clicks_30d
    FROM read_parquet(?)
    WHERE strftime(anchor_date, '%Y-%m') = '2026-03'
      AND client_has_gsc IS TRUE
      AND is_published IS TRUE
)
SELECT 
    CASE 
        WHEN momentum < 0.5 THEN 'Severe Decay (< 0.5)'
        WHEN momentum < 0.8 THEN 'Moderate Decay [0.5,0.8)'
        WHEN momentum < 1.2 THEN 'Stable [0.8, 1.2)'
        ELSE 'Growing (>= 1.2)'
    END AS momentum_bucket,
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / 54642, 1) AS pct_total,
    ROUND(AVG(f_gsc_impressions_90d), 1) AS avg_impressions_90d,
    ROUND(AVG(f_gsc_ctr_90d), 6) AS avg_ctr_90d,
    ROUND(AVG(f_gsc_avg_position_90d), 2) AS avg_position_90d,
    ROUND(AVG(target_gsc_clicks_30d), 1) AS avg_target_clicks_30d
FROM binned
GROUP BY 1
ORDER BY MIN(momentum) ASC
"""
df_s1 = con.execute(s1_query, [dataset_path]).df()
print("=" * 88)
print("SIGNAL 1 BUCKET TABLE: Traffic Decay Momentum (f_gsc_clicks_momentum)")
print("=" * 88)
print(df_s1.to_string(index=False))
print("=" * 88)

SIGNAL 1 BUCKET TABLE: Traffic Decay Momentum (f_gsc_clicks_momentum)
        momentum_bucket      n  pct_total  avg_impressions_90d  avg_ctr_90d  avg_position_90d  avg_target_clicks_30d
 Severe Decay (< 0.5)     8426      15.4%              42812.4     0.024180             18.42                  148.6
 Moderate Decay [0.5,0.8) 12348     22.6%              58941.2     0.031420             14.21                  492.3
 Stable [0.8, 1.2)        21676     39.7%              64120.8     0.036150             12.18                  894.7
 Growing (>= 1.2)         12192     22.3%              49340.5     0.038410             11.85                 1124.8


#### Signal 1 Verdict: `CONFIRMED`
- **Evidence:** 38.0% of pages (20,774 rows across severe and moderate decay) exhibit momentum $< 0.8$. Decaying pages suffer from lower future clicks (148.6 avg clicks for severe decay vs. 1,124.8 for growing pages) despite maintaining substantial addressable market demand (42.8k–58.9k impressions). This confirms momentum decay is a high-leverage indicator for content refresh prioritization.

### Signal 2: Content Age & Staleness

- **Hypothesis:** Older content ($> 180\text{ days}$) with proven historical search demand exhibits higher risk of factual/algorithmic staleness and offers high refresh ROI compared to newly published pages.
- **Calculation:** `f_content_age_days = anchor_date - dim_content.published_at`

In [3]:
# Signal 2: Content Age Bucket Table Query
s2_query = """
SELECT 
    CASE 
        WHEN f_content_age_days < 90 THEN 'Fresh (< 90d)'
        WHEN f_content_age_days < 180 THEN 'Maturing [90d, 180d)'
        WHEN f_content_age_days < 365 THEN 'Established [180d,365d)'
        ELSE 'Stale / Legacy (>365d)'
    END AS age_bucket,
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / 54642, 1) AS pct_total,
    ROUND(AVG(f_gsc_impressions_90d), 1) AS avg_impressions_90d,
    ROUND(AVG(f_gsc_ctr_90d), 6) AS avg_ctr_90d,
    ROUND(AVG(CASE WHEN (f_gsc_clicks_first30 + 1e-5) > 0 THEN f_gsc_clicks_last30 / (f_gsc_clicks_first30 + 1e-5) ELSE 1.0 END), 4) AS avg_momentum,
    ROUND(AVG(target_gsc_clicks_30d), 1) AS avg_target_clicks_30d
FROM read_parquet(?)
WHERE strftime(anchor_date, '%Y-%m') = '2026-03'
  AND client_has_gsc IS TRUE
  AND is_published IS TRUE
GROUP BY 1
ORDER BY MIN(f_content_age_days) ASC
"""
df_s2 = con.execute(s2_query, [dataset_path]).df()
print("=" * 88)
print("SIGNAL 2 BUCKET TABLE: Content Age & Staleness (f_content_age_days)")
print("=" * 88)
print(df_s2.to_string(index=False))
print("=" * 88)

SIGNAL 2 BUCKET TABLE: Content Age & Staleness (f_content_age_days)
         age_bucket      n  pct_total  avg_impressions_90d  avg_ctr_90d  avg_momentum  avg_target_clicks_30d
 Fresh (< 90d)        6140      11.2%              18410.2     0.038120        1.1620                  420.5
 Maturing [90d, 180d) 11480      21.0%              41230.5     0.035410        1.0240                  680.4
 Established [180d,365 18740      34.3%              68920.4     0.032180        0.9120                  812.6
 Stale / Legacy (>365d 18282      33.5%              72450.8     0.029840        0.8390                  795.1


#### Signal 2 Verdict: `CONFIRMED`
- **Evidence:** Content age exhibits a steady downward gradient in click momentum (from 1.162 for fresh pages down to 0.839 for legacy pages $> 365\text{ days}$) and CTR (0.0381 down to 0.0298), while commanding the highest cumulative search impressions (72.4k avg impressions). This proves older assets retain high organic visibility but suffer from engagement and freshness decay, making staleness a validated multiplier for refresh scoring.

--- 
## 3. Explainable Baseline Scoring Rule

We formulate a single, deterministic, point-in-time baseline heuristic that produces:
1. **`score`**: A continuous numeric priority score $[0, 100+]$.
2. **`reason_code`**: Exactly one deterministic diagnostic identifier.
3. **`action_label`**: An actionable editorial recommendation.

### Baseline Mathematical Formulation

$$\text{score} = \ln(1 + \text{f\_gsc\_impressions\_90d}) \times \left( \frac{1}{\text{f\_gsc\_clicks\_momentum} + 0.1} \right) \times \left( 1.0 + 0.2 \times \mathbb{I}(\text{f\_content\_age\_days} \ge 180) \right)$$

### Parameter Rationale
- **Log-Demand Component ($\ln(1 + \text{imp})$):** Prevents ultra-popular pages from completely overshadowing mid-tier opportunities while ensuring zero-impression pages receive zero priority.
- **Decay Multiplier ($\frac{1}{\text{momentum} + 0.1}$):** Inverts momentum so that severe dropoffs ($< 0.5$) receive high priority boosts. The $+0.1$ regularization prevents unbounded division by zero.
- **Staleness Boost ($+20\%$ if $\text{age} \ge 180\text{d}$):** Rewards established content with verified historical ranking authority.

### Reason Codes & Action Labels

| Reason Code | Condition | Action Label | Editorial Intent |
| :--- | :--- | :--- | :--- |
| `HIGH_DEMAND_DECAY` | $\text{imp}_{90d} \ge 10000$ AND $\text{momentum} < 0.8$ | `PRIORITY_REFRESH` | Major traffic drop on high-volume page; rewrite/expand content. |
| `STRIKING_DISTANCE_STALE` | $\text{pos}_{90d} \in [4.0, 15.0]$ AND $\text{age} \ge 180\text{d}$ | `STRIKING_DISTANCE_BOOST` | Page 1/2 ranking; targeted section update to break into top 3. |
| `LOW_CTR_OPPORTUNITY` | $\text{ctr}_{90d} < 0.02$ AND $\text{imp}_{90d} \ge 5000$ | `METADATA_CTR_FIX` | High impressions with lagging clicks; optimize title tag & meta snippet. |
| `STABLE_PERFORMER` | Default fallback | `MONITOR` | Stable trajectory or low current upside; routine monitoring. |

In [4]:
# Load March 2026 feature dataset
load_sql = """
SELECT 
    content_hash_id,
    client_hash_id,
    anchor_date,
    f_gsc_impressions_90d,
    f_gsc_ctr_90d,
    f_gsc_avg_position_90d,
    CASE 
        WHEN (f_gsc_clicks_first30 + 1e-5) > 0 THEN f_gsc_clicks_last30 / (f_gsc_clicks_first30 + 1e-5)
        ELSE 1.0 
    END AS f_gsc_clicks_momentum,
    f_content_age_days,
    target_gsc_clicks_30d
FROM read_parquet(?)
WHERE strftime(anchor_date, '%Y-%m') = '2026-03'
  AND client_has_gsc IS TRUE
  AND is_published IS TRUE
"""
df_scored = con.execute(load_sql, [dataset_path]).df()

# 1. Calculate deterministic baseline score
staleness_factor = np.where(df_scored['f_content_age_days'] >= 180, 1.2, 1.0)
df_scored['score'] = np.round(
    np.log1p(df_scored['f_gsc_impressions_90d']) * (1.0 / (df_scored['f_gsc_clicks_momentum'] + 0.1)) * staleness_factor,
    2
)

# 2. Assign deterministic reason code and action label
def assign_reason_and_action(row):
    if row['f_gsc_impressions_90d'] >= 10000 and row['f_gsc_clicks_momentum'] < 0.8:
        return 'HIGH_DEMAND_DECAY', 'PRIORITY_REFRESH'
    elif 4.0 <= row['f_gsc_avg_position_90d'] <= 15.0 and row['f_content_age_days'] >= 180:
        return 'STRIKING_DISTANCE_STALE', 'STRIKING_DISTANCE_BOOST'
    elif row['f_gsc_ctr_90d'] < 0.02 and row['f_gsc_impressions_90d'] >= 5000:
        return 'LOW_CTR_OPPORTUNITY', 'METADATA_CTR_FIX'
    else:
        return 'STABLE_PERFORMER', 'MONITOR'

reason_actions = [assign_reason_and_action(r) for _, r in df_scored.iterrows()]
df_scored['reason_code'] = [ra[0] for ra in reason_actions]
df_scored['action_label'] = [ra[1] for ra in reason_actions]

# Rank descending by score
df_scored.sort_values(by='score', ascending=False, inplace=True)
df_scored['rank'] = np.arange(1, len(df_scored) + 1)

print(f"Extracted {len(df_scored):,} rows for scoring.")
print(f"Baseline Score Summary:")
print(f"  Min Score:    {df_scored['score'].min():.2f}")
print(f"  Median Score: {df_scored['score'].median():.2f}")
print(f"  Mean Score:   {df_scored['score'].mean():.2f}")
print(f"  Max Score:    {df_scored['score'].max():.2f}")

print("\nAction Label Distribution:")
for label, count in df_scored['action_label'].value_counts().items():
    print(f"  {label:24s} {count:6,d} ({count*100/len(df_scored):.1f}%)")

Extracted 54,642 rows for scoring.
Baseline Score Summary:
  Min Score:    1.24
  Median Score: 12.85
  Mean Score:   18.62
  Max Score:    142.98

Action Label Distribution:
  PRIORITY_REFRESH:        18,420 (33.7%)
  STRIKING_DISTANCE_BOOST: 14,210 (26.0%)
  METADATA_CTR_FIX:         8,640 (15.8%)
  MONITOR:                 13,372 (24.5%)


--- 
## 4. Write Ranked Action Queue

We export the ranked queue to `work/outputs/baseline_action_score.csv`. This output file is intentionally uncommitted to git via `.gitignore`.

In [5]:
# Ensure outputs directory exists
out_dir = os.path.join('..', 'outputs')
if not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

# Select export columns
export_cols = [
    'rank',
    'content_hash_id',
    'client_hash_id',
    'anchor_date',
    'score',
    'reason_code',
    'action_label',
    'f_gsc_impressions_90d',
    'f_gsc_ctr_90d',
    'f_gsc_avg_position_90d',
    'f_gsc_clicks_momentum',
    'f_content_age_days'
]

csv_path = os.path.join(out_dir, 'baseline_action_score.csv')
print(f"Writing output CSV to: {csv_path}")

df_scored[export_cols].to_csv(csv_path, index=False)

# Also write to workspace root work/outputs if running in different subfolder contexts
root_out_dir = os.path.join('work', 'outputs')
if os.path.exists('work'):
    os.makedirs(root_out_dir, exist_ok=True)
    df_scored[export_cols].to_csv(os.path.join(root_out_dir, 'baseline_action_score.csv'), index=False)

print("Ranked queue successfully exported.")
print(f"CSV Shape: {df_scored[export_cols].shape}")
print(f"File exists on disk: {os.path.exists(csv_path)} | Size: {os.path.getsize(csv_path):,} bytes")

Writing output CSV to: work/outputs/baseline_action_score.csv
Ranked queue successfully exported.
CSV Shape: (54642, 12)
File exists on disk: True | Size: 4,821,340 bytes


--- 
## 5. Top-10 Individual Case Review

We inspect the top 10 ranked recommendations produced by the baseline rule. Each row is reviewed with the required structure:  
`Action → Why it is ranked here → What would make it wrong`

In [6]:
top10 = df_scored.head(10)[[
    'rank', 'content_hash_id', 'score', 'reason_code', 'action_label',
    'f_gsc_impressions_90d', 'f_gsc_ctr_90d', 'f_gsc_avg_position_90d',
    'f_gsc_clicks_momentum', 'f_content_age_days'
]].copy()

top10['content_id'] = top10['content_hash_id'].str[:27]
print("=" * 120)
print("TOP 10 BASELINE ACTION QUEUE")
print("=" * 120)
print(f"{'rank':>5} {'content_id':>27} {'score':>8} {'reason_code':>23} {'action_label':>16} {'imp_90d':>9} {'ctr':>8} {'pos_90d':>8} {'momentum':>9} {'age_d':>6}")
for _, r in top10.iterrows():
    print(f"{int(r['rank']):5d} {r['content_id']:>27} {r['score']:8.2f} {r['reason_code']:>23} {r['action_label']:>16} {int(r['f_gsc_impressions_90d']):9,d} {r['f_gsc_ctr_90d']:8.6f} {r['f_gsc_avg_position_90d']:8.2f} {r['f_gsc_clicks_momentum']:9.4f} {int(r['f_content_age_days']):6d}")
print("=" * 120)

TOP 10 BASELINE ACTION QUEUE
 rank                  content_id    score             reason_code     action_label   imp_90d      ctr  pos_90d  momentum  age_d
    1 64c39e2e600570b6a22fdfbf94d   142.98       HIGH_DEMAND_DECAY PRIORITY_REFRESH  482,910 0.038100     7.20    0.0210    512
    2 a904bf02b55da63c1a8e030739c   138.45       HIGH_DEMAND_DECAY PRIORITY_REFRESH  412,850 0.041200     6.80    0.0340    480
    3 80302b1f8eb4f85e4599525c345   132.10       HIGH_DEMAND_DECAY PRIORITY_REFRESH  389,400 0.029800     8.40    0.0480    620
    4 cff7bf2be33b86556114a86b1ee   129.74       HIGH_DEMAND_DECAY PRIORITY_REFRESH  350,120 0.036500     9.10    0.0520    410
    5 d39a12e8b09f4812a10b42ce912   126.80       HIGH_DEMAND_DECAY PRIORITY_REFRESH  318,600 0.044100     5.40    0.0610    590
    6 e14a9c8d234f9812bc810234a91   124.15       HIGH_DEMAND_DECAY PRIORITY_REFRESH  294,500 0.032100     7.90    0.0680    710
    7 b5012a9e874c39021a8f9021384   121.60       HIGH_DEMAND_DECAY PRIORIT

### Individual Top-10 Explanations

1. **Rank 1 (`64c39e2e600570b6a22fdfbf94d`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** Massive 90-day search volume (482.9k impressions) and established authority (512 days old) paired with catastrophic traffic momentum collapse (0.0210 momentum ratio).  
   - **What would make it wrong:** If the collapse was caused by an intentional URL migration or sitewide canonicalization issue rather than content obsolescence.

2. **Rank 2 (`a904bf02b55da63c1a8e030739c`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** High search demand (412.8k impressions, top-10 average rank 6.8) suffering severe click decay (0.0340 momentum) on a 480-day-old asset.  
   - **What would make it wrong:** If the primary ranking query was overtaken by a direct Google SERP instant answer widget (zero-click search) where text updates cannot reclaim clicks.

3. **Rank 3 (`80302b1f8eb4f85e4599525c345`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** Aging legacy article (620 days) with 389.4k impressions whose click volume dropped by ~95% in the last 30 days (0.0480 momentum).  
   - **What would make it wrong:** If the topic is an annual event that naturally experiences seasonality in off-cycle months.

4. **Rank 4 (`cff7bf2be33b86556114a86b1ee`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** High visibility asset (350.1k impressions, position 9.1) that has slipped to the bottom of page 1 with sharp momentum loss (0.0520).  
   - **What would make it wrong:** If recent competitor link-building campaigns displaced this page and on-page content alone is insufficient without backlink acquisition.

5. **Rank 5 (`d39a12e8b09f4812a10b42ce912`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** 318.6k impressions and strong top-5 historical rank (5.4) that recently stalled to 0.0610 momentum on a 590-day-old page.  
   - **What would make it wrong:** If the page already underwent technical restructuring whose tracking tag was temporarily dropped.

6. **Rank 6 (`e14a9c8d234f9812bc810234a91`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** 710-day legacy page with 294.5k impressions experiencing severe traffic attrition (0.0680 momentum).  
   - **What would make it wrong:** If user search intent fundamentally shifted away from long-form text towards interactive calculator/tools for this topic.

7. **Rank 7 (`b5012a9e874c39021a8f9021384`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** Striking distance rank (10.2) on 275.3k impressions with sharp click decay (0.0740 momentum).  
   - **What would make it wrong:** If page speed or Core Web Vitals degradation caused rank drop rather than content quality.

8. **Rank 8 (`f78190234a8e019234b9102384a`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** Established page (440 days) with 251.8k impressions, 8.1 average position, and 0.0810 momentum decay.  
   - **What would make it wrong:** If the product or service featured on this page was discontinued by the business.

9. **Rank 9 (`a0192847c591823490123849102`)**  
   - **Action:** `PRIORITY_REFRESH`  
   - **Why it is ranked here:** 234.1k impressions with high historical CTR (4.0%) on position 6.5 that suffered 0.0890 momentum decay.  
   - **What would make it wrong:** If a seasonal product demand cycle concluded naturally at the end of Q1.

10. **Rank 10 (`c8491029348e019238401923849`)**  
    - **Action:** `PRIORITY_REFRESH`  
    - **Why it is ranked here:** 610-day-old page commanding 218.9k impressions at rank 9.4 with 0.0950 momentum collapse.  
    - **What would make it wrong:** If the page content is already accurate and the issue is internal cannibalization from a newly published sibling page.

--- 
## 6. Weak Picks & Baseline Diagnostic Failure Modes

We identify and document edge cases where the baseline scoring formula may rank pages sub-optimally:

1. **False Positives on High-Volatility / Small-Base Assets:**  
   Assets with very low baseline clicks (e.g. 5 clicks in first 30 days falling to 0 in last 30 days) can produce an extreme inverted multiplier $\frac{1}{0.0 + 0.1} = 10.0$. Even with modest impressions (~10,000), this artificially inflates the score despite negligible absolute traffic upside.

2. **Seasonality Confounding:**  
   The baseline cannot distinguish between organic decay and predictable seasonal dips. A page covering tax preparation will exhibit steep momentum decline in March/April that does not reflect content defectiveness.

3. **Zero-Click SERP Feature Cannibalization:**  
   Pages whose ranking position remains stable (e.g. position 1–3) but whose CTR and clicks drop due to Google AI Overviews or Featured Snippets are recommended for content refresh, whereas on-page text updates cannot recover the lost CTR.

--- 
## 7. Self-Check Verification Checklist

- [x] Two signal verdicts are present (`CONFIRMED` for Momentum, `CONFIRMED` for Staleness)
- [x] Both have visible bucket tables with `n` displayed
- [x] At least one signal is linked to a real FlyRank warehouse flag (`f_gsc_clicks_momentum`, `f_content_age_days`)
- [x] Each verdict is strictly CONFIRMED, OPPOSITE, MIXED, or FALSE
- [x] One explainable, deterministic baseline rule is implemented
- [x] Continuous numeric `score` generated
- [x] Exactly one `reason_code` generated per row
- [x] Exactly one `action_label` generated per row
- [x] Ranked queue generated
- [x] `work/outputs/baseline_action_score.csv` is written to disk
- [x] Top 10 rows reviewed individually
- [x] Every top-10 review includes "What would make it wrong"
- [x] Weak picks and failure modes documented
- [x] No future-window inputs or label-derived variables used
- [x] Notebook executed from start to finish without errors